# 🎵 ALM — Audio Encoder Training Notebook
**Project:** Audio-Language Model (ALM)  
**Component:** Audio Encoder (CNN + Transformer)    
**CUDA:** 12.1

---
> **Data structure expected:**
> ```
> data/
>   processed/
>     audio/       ← .npy or .wav files
>     metadata.csv ← columns: filename, label (or transcript)
>   raw/
> ```

## 1️⃣  Imports & CUDA Setup

In [1]:
import os
import sys
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchaudio
import torchaudio.transforms as T
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── CUDA Check ────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 55)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.version.cuda}")
print(f"  Device   : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU      : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  VRAM     : {vram:.1f} GB")
print("=" * 55)

  PyTorch  : 2.5.1+cu121
  CUDA     : 12.1
  Device   : cuda
  GPU      : NVIDIA GeForce RTX 2050
  VRAM     : 4.0 GB


## 2️⃣  Configuration

In [2]:
class CFG:
    # ── Paths ─────────────────────────────────────────────────────────────────
    DATA_DIR      = Path(r"C:\Users\shara\Desktop\ALM\data\processed")
    AUDIO_DIR     = DATA_DIR / "audio"
    METADATA_PATH = DATA_DIR / "metadata.csv"
    MODEL_DIR     = Path(r"C:\Users\shara\Desktop\ALM\models")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    # ── Audio — UrbanSound8K preprocessed spec ────────────────────────────────
    SAMPLE_RATE   = 24_000
    DURATION      = 5
    NUM_SAMPLES   = 120_000
    N_MELS        = 128
    N_FFT         = 1024
    HOP_LENGTH    = 512
    F_MIN         = 0
    F_MAX         = 8_000

    # ── Model ─────────────────────────────────────────────────────────────────
    EMBED_DIM     = 256
    NUM_HEADS     = 8
    NUM_LAYERS    = 4
    FF_DIM        = 1024
    DROPOUT       = 0.1

    # ── Training ──────────────────────────────────────────────────────────────
    EPOCHS        = 50
    BATCH_SIZE    = 16     # Keep small for 4 GB VRAM
    LR            = 3e-4
    WEIGHT_DECAY  = 1e-4
    VAL_FOLD      = 9      # fold 9 → validation
    TEST_FOLD     = 10     # fold 10 → test
    GRAD_CLIP     = 1.0
    AMP           = True   # Mixed-precision (FP16) — halves VRAM usage
    PATIENCE      = 10     # Early stopping

    DEVICE        = DEVICE

print("✅ Config loaded")
print(f"   Audio dir    : {CFG.AUDIO_DIR}")
print(f"   Metadata     : {CFG.METADATA_PATH}")
print(f"   Model output : {CFG.MODEL_DIR}")
print(f"   Sample rate  : {CFG.SAMPLE_RATE} Hz | {CFG.DURATION}s | {CFG.NUM_SAMPLES} samples")
print(f"   Val fold: {CFG.VAL_FOLD}  |  Test fold: {CFG.TEST_FOLD}")

✅ Config loaded
   Audio dir    : C:\Users\shara\Desktop\ALM\data\processed\audio
   Metadata     : C:\Users\shara\Desktop\ALM\data\processed\metadata.csv
   Model output : C:\Users\shara\Desktop\ALM\models
   Sample rate  : 24000 Hz | 5s | 120000 samples
   Val fold: 9  |  Test fold: 10


## 3️⃣  Audio Encoder Model  
*(Same architecture from `utils/audio_encoder.py` — imported inline so the notebook is self-contained)*

In [3]:
# ── Add project root to path so we can import from utils/ ────────────────────
PROJECT_ROOT = Path(r"C:\Users\shara\Desktop\ALM").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from utils.audio_encoder import AudioEncoder
    print("✅ AudioEncoder imported from utils/audio_encoder.py")
    IMPORTED = True
except ImportError:
    print("⚠️  Could not import — defining AudioEncoder inline (fallback)")
    IMPORTED = False

# from utils.audio_encoder import AudioEncoder

✅ AudioEncoder imported from utils/audio_encoder.py


## 4️⃣  Dataset

In [4]:
# ── Add script/ folder to path ────────────────────────────────────────────────
import logging
log_dir = PROJECT_ROOT / 'outputs' / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
logging.basicConfig(filename=log_dir / 'training.log', level=logging.INFO)
logger = logging.getLogger(__name__)

from script.audio_dataset import AudioTextDataset

In [ ]:
# ── Train / Val / Test splits by fold ─────────────────────────────────────────
train_loader, val_loader, label2id = AudioTextDataset.get_loaders(
    metadata_csv = CFG.METADATA_PATH,
    audio_dir    = CFG.AUDIO_DIR,
    val_fold     = CFG.VAL_FOLD,       # fold 9 = val
    batch_size   = CFG.BATCH_SIZE,
    num_workers  = 4,
    augment      = True,
)

# ── Separate test loader (fold 10) ────────────────────────────────────────────
test_ds = AudioTextDataset(
    CFG.METADATA_PATH, CFG.AUDIO_DIR,
    split='test', val_fold=CFG.TEST_FOLD, augment=False
)
test_ds.label2id    = label2id
test_ds.id2label    = {v: k for k, v in label2id.items()}
test_ds.num_classes = len(label2id)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE * 2,
                         shuffle=False, num_workers=4, pin_memory=True)

NUM_CLASSES = len(label2id)
id2label    = {v: k for k, v in label2id.items()}

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'Classes ({NUM_CLASSES}): {list(label2id.keys())}')

[AudioTextDataset] split='train'  samples=9673  classes=58  val_fold=9
[AudioTextDataset] split='val'  samples=1059  classes=57  val_fold=9
[AudioTextDataset] split='test'  samples=1078  classes=58  val_fold=10
Train batches : 604
Val batches   : 34
Test batches  : 34
Classes (58): ['air_conditioner', 'airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn', 'cat', 'chainsaw', 'children_playing', 'chirping_birds', 'church_bells', 'clapping', 'clock_alarm', 'clock_tick', 'coughing', 'cow', 'crackling_fire', 'crickets', 'crow', 'crying_baby', 'dog', 'dog_bark', 'door_wood_creaks', 'door_wood_knock', 'drilling', 'drinking_sipping', 'engine', 'engine_idling', 'fireworks', 'footsteps', 'frog', 'glass_breaking', 'gun_shot', 'hand_saw', 'helicopter', 'hen', 'insects', 'jackhammer', 'keyboard_typing', 'laughing', 'mouse_click', 'pig', 'pouring_water', 'rain', 'rooster', 'sea_waves', 'sheep', 'siren', 'sneezing', 'snoring', 'street_music', 'thunderstorm', 'toilet_flush', 'train', 'v

## 6️⃣  Initialise Model, Optimizer & Scheduler

In [11]:
model = AudioEncoder(
    embed_dim=CFG.EMBED_DIM,
    dropout=CFG.DROPOUT,
    num_classes=NUM_CLASSES
).to(CFG.DEVICE)

# ── Parameter count ───────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

# ── Loss, Optimizer, Scheduler ────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY,
    betas=(0.9, 0.999),
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-6
)

# ── Mixed-precision scaler (AMP) ──────────────────────────────────────────────
scaler = GradScaler(enabled=CFG.AMP)

print("\n✅ Model, optimizer & scheduler ready")
print(f"   AMP (FP16) : {'enabled ✓' if CFG.AMP else 'disabled'}")
print(f"   Device     : {CFG.DEVICE}")

Total params    : 5,647,322
Trainable params: 5,647,322

✅ Model, optimizer & scheduler ready
   AMP (FP16) : enabled ✓
   Device     : cuda


## 7️⃣  Training & Validation Loop

In [12]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(loader, desc="  Train", leave=False):
        mels   = batch['mel'].to(device, non_blocking=True)
        labels = batch['label_id'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=CFG.AMP):
            logits = model(mels)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * mels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += mels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(loader, desc="  Val  ", leave=False):
        mels   = batch['mel'].to(device, non_blocking=True)
        labels = batch['label_id'].to(device, non_blocking=True)

        with autocast(enabled=CFG.AMP):
            logits = model(mels)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * mels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += mels.size(0)

    return total_loss / total, correct / total


print("✅ Training functions defined")

✅ Training functions defined


In [13]:
# ── History containers ────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'lr': []
}

best_val_loss  = float('inf')
patience_count = 0
BEST_MODEL_PATH = CFG.MODEL_DIR / "alm_audio_encoder_best.pth"

print(f"  Starting training for {CFG.EPOCHS} epochs on {CFG.DEVICE}")
print("-" * 65)
print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>9}  {'Val Acc':>8}  {'LR':>9}")
print("-" * 65)

start_time = time.time()

for epoch in range(1, CFG.EPOCHS + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, CFG.DEVICE)
    v_loss, v_acc = evaluate(model, val_loader, criterion, CFG.DEVICE)

    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    history['lr'].append(current_lr)

    # ── Checkpoint best model ─────────────────────────────────────────────────
    if v_loss < best_val_loss:
        best_val_loss  = v_loss
        patience_count = 0
        torch.save({
            'epoch':       epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss':    v_loss,
            'val_acc':     v_acc,
            'config': {
                'n_mels': CFG.N_MELS, 'embed_dim': CFG.EMBED_DIM,
                'num_heads': CFG.NUM_HEADS, 'num_layers': CFG.NUM_LAYERS,
                'ff_dim': CFG.FF_DIM, 'num_classes': NUM_CLASSES,
            },
            'label_classes': list(label2id.keys()),
        }, BEST_MODEL_PATH)
        flag = " ←  saved"
    else:
        patience_count += 1
        flag = ""

    print(f"{epoch:>6}  {t_loss:>10.4f}  {t_acc*100:>8.2f}%  {v_loss:>9.4f}  {v_acc*100:>7.2f}%  {current_lr:>9.2e}{flag}")

    # ── Early stopping ────────────────────────────────────────────────────────
    if patience_count >= CFG.PATIENCE:
        print(f"\n⏹  Early stopping triggered at epoch {epoch}")
        break

elapsed = time.time() - start_time
print("-" * 65)
print(f"\n✅ Training complete in {elapsed/60:.1f} min")
print(f"   Best val loss : {best_val_loss:.4f}")
print(f"   Model saved   : {BEST_MODEL_PATH}")

  Starting training for 50 epochs on cuda
-----------------------------------------------------------------
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc         LR
-----------------------------------------------------------------


     1      3.2827     30.05%     2.8415    34.18%   3.00e-04 ←  saved


     2      2.6211     43.04%     2.4696    48.44%   3.00e-04 ←  saved


     3      2.2912     53.00%     2.1179    63.46%   2.99e-04 ←  saved


     4      2.0674     59.04%     1.9128    64.21%   2.97e-04 ←  saved


...

    31      0.9745     94.43%     1.0587    90.65%   1.04e-04


    32      0.9553     95.24%     1.0275    91.88%   9.55e-05 ←  saved


    33      0.9492     95.41%     1.0126    92.63%   8.68e-05 ←  saved


    34      0.9435     95.83%     1.0098    92.63%   7.85e-05 ←  saved


KeyboardInterrupt: 

## 8️⃣  Save Final (Last) Model

In [ ]:
FINAL_MODEL_PATH = CFG.MODEL_DIR / "alm_audio_encoder_final.pth"

torch.save({
    'model_state':   model.state_dict(),
    'optim_state':   optimizer.state_dict(),
    'history':       history,
    'label_classes': list(label2id.keys()),
    'config': {
        'n_mels': CFG.N_MELS, 'embed_dim': CFG.EMBED_DIM,
        'num_heads': CFG.NUM_HEADS, 'num_layers': CFG.NUM_LAYERS,
        'ff_dim': CFG.FF_DIM, 'num_classes': NUM_CLASSES,
    },
}, FINAL_MODEL_PATH)

print(f"✅ Final model saved → {FINAL_MODEL_PATH}")

# ── Models directory listing ──────────────────────────────────────────────────
print("\nModels directory:")
for f in sorted(CFG.MODEL_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name:45s}  {size_mb:.1f} MB")

## 9️⃣  Training & Validation Loss / Accuracy Plots

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0f0f1a')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

PLOT_STYLE = dict(linewidth=2.2)
GRID_STYLE = dict(alpha=0.15, linestyle='--', color='white')

# ── Helper ────────────────────────────────────────────────────────────────────
def style_ax(ax, title):
    ax.set_facecolor('#1a1a2e')
    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)
    ax.tick_params(colors='#aaaacc')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')
    ax.grid(**GRID_STYLE)
    ax.legend(framealpha=0.15, labelcolor='white', edgecolor='#444466')

# ── Loss ──────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs_ran, history['train_loss'], color='#4fc3f7', label='Train Loss', **PLOT_STYLE)
ax1.plot(epochs_ran, history['val_loss'],   color='#ef9a9a', label='Val Loss',   **PLOT_STYLE)
best_ep = history['val_loss'].index(min(history['val_loss'])) + 1
ax1.axvline(best_ep, color='#ffd54f', linestyle=':', linewidth=1.5, label=f'Best (ep {best_ep})')
ax1.set_xlabel('Epoch', color='#aaaacc')
ax1.set_ylabel('Loss',  color='#aaaacc')
style_ax(ax1, '📉 Loss Curve')

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs_ran, [a*100 for a in history['train_acc']], color='#a5d6a7', label='Train Acc', **PLOT_STYLE)
ax2.plot(epochs_ran, [a*100 for a in history['val_acc']],   color='#ce93d8', label='Val Acc',   **PLOT_STYLE)
ax2.set_xlabel('Epoch',    color='#aaaacc')
ax2.set_ylabel('Accuracy %', color='#aaaacc')
style_ax(ax2, '🎯 Accuracy Curve')

# ── LR Schedule ───────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(epochs_ran, history['lr'], color='#ffcc80', **PLOT_STYLE)
ax3.set_xlabel('Epoch', color='#aaaacc')
ax3.set_ylabel('LR',    color='#aaaacc')
ax3.set_yscale('log')
style_ax(ax3, '📈 Learning Rate Schedule')

# ── Loss difference (overfitting monitor) ─────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
gap = [v - t for t, v in zip(history['train_loss'], history['val_loss'])]
ax4.fill_between(epochs_ran, gap, alpha=0.4, color='#ef9a9a')
ax4.plot(epochs_ran, gap, color='#ef9a9a', **PLOT_STYLE)
ax4.axhline(0, color='white', linewidth=1, linestyle='-')
ax4.set_xlabel('Epoch', color='#aaaacc')
ax4.set_ylabel('Val − Train Loss', color='#aaaacc')
style_ax(ax4, '🔍 Generalisation Gap')

fig.suptitle('ALM — Audio Encoder Training Dashboard', 
             color='white', fontsize=15, fontweight='bold', y=1.01)

PLOT_PATH = PROJECT_ROOT/ 'output'/'figures'/ "training_curves.png"
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"✅ Plot saved → {PLOT_PATH}")

## 🔟  Test Set Evaluation & Accuracy

In [ ]:
# ── Load best model ───────────────────────────────────────────────────────────
ckpt = torch.load(BEST_MODEL_PATH, map_location=CFG.DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f"✅ Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")

# ── Inference on test set ─────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        mels = batch['mel'].to(CFG.DEVICE, non_blocking=True)
        labels = batch['label_id'].to(CFG.DEVICE, non_blocking=True)
        with autocast(enabled=CFG.AMP):
            logits = model(mels)
        preds = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = (all_preds == all_labels).mean() * 100
print(f"\n{'='*45}")
print(f"  Test Accuracy : {test_acc:.2f}%")
print(f"{'='*45}")

# ── Classification report ─────────────────────────────────────────────────────
print("\nPer-class Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=list(le.classes_),
    digits=4
))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f0f1a')

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Confusion Matrix (Counts)', 'Confusion Matrix (Normalised)']
):
    ax.set_facecolor('#1a1a2e')
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=label2id.keys(), yticklabels=label2id.keys(),
        ax=ax, linewidths=0.4, linecolor='#222240',
        annot_kws={'size': 9}
    )
    ax.set_title(title,     color='white', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', color='#aaaacc')
    ax.set_ylabel('Actual',    color='#aaaacc')
    ax.tick_params(colors='#aaaacc')

fig.suptitle(f'ALM Audio Encoder — Test Accuracy: {test_acc:.2f}%',
             color='white', fontsize=14, fontweight='bold')
CM_PATH = CFG.MODEL_DIR / "confusion_matrix.png"
plt.savefig(CM_PATH, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"✅ Confusion matrix saved → {CM_PATH}")

## 1️⃣1️⃣  How to Load & Use the Saved Model

In [ ]:
# ── Inference example ─────────────────────────────────────────────────────────
def load_alm_encoder(ckpt_path: str, device: str = 'cuda'):
    """Load ALM AudioEncoder from checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=device)
    cfg  = ckpt['config']
    enc  = AudioEncoder(
        n_mels      = cfg['n_mels'],
        embed_dim   = cfg['embed_dim'],
        num_heads   = cfg['num_heads'],
        num_layers  = cfg['num_layers'],
        ff_dim      = cfg['ff_dim'],
        num_classes = cfg['num_classes'],
    ).to(device)
    enc.load_state_dict(ckpt['model_state'])
    enc.eval()
    label_classes = ckpt.get('label_classes', [])
    print(f"✅ Loaded encoder  |  classes: {label_classes}")
    return enc, label_classes


# Demo — get embedding for a random batch
encoder, classes = load_alm_encoder(str(BEST_MODEL_PATH), device=str(CFG.DEVICE))

dummy = torch.randn(1, 1, CFG.N_MELS, 94).to(CFG.DEVICE)  # 94 ≈ 3s at 16kHz/512
with torch.no_grad():
    logit = encoder(dummy)
    emb   = encoder.encode(dummy)

print(f"Logit shape    : {logit.shape}")
print(f"Embedding shape: {emb.shape}")

---
## 📦  Summary

| Artifact | Path |
|---|---|
| Best model (lowest val loss) | `models/alm_audio_encoder_best.pth` |
| Final model (last epoch) | `models/alm_audio_encoder_final.pth` |
| Training curves plot | `models/training_curves.png` |
| Confusion matrix plot | `models/confusion_matrix.png` |

**References:**
- SpecAugment: [Park et al., 2019](https://arxiv.org/abs/1904.08779)
- Audio Spectrogram Transformer (AST): [Gong et al., 2021](https://arxiv.org/abs/2104.01778)
- Pre-LN Transformer: [Xiong et al., 2020](https://arxiv.org/abs/2002.04745)
- AdamW + Cosine LR: [Loshchilov & Hutter, 2019](https://arxiv.org/abs/1711.05101)
- Mixed Precision Training: [Micikevicius et al., 2018](https://arxiv.org/abs/1710.03740)